# StressDetection — RL Retrain on Google Colab

Use this notebook when you want to **fine-tune the existing model checkpoint
with user feedback** collected by your running API — without needing a GPU
on your local machine.

### What it does
1. Clones the repo and installs dependencies.
2. Lets you **upload** your `stress_detection.db` and `model.pt` from your computer.
3. Optionally loads the original `unified_stress.csv` from Google Drive to prevent
   catastrophic forgetting.
4. Runs `training/retrain.py` on a **free T4 GPU** (≈ 2–5 minutes).
5. Saves the updated `model.pt` to Google Drive **and** lets you download it.

### Before you start
1. Click **Runtime → Change runtime type** and set the hardware accelerator to **T4 GPU**.
2. Have these two files handy on your local machine:
   - `stress_detection.db` — your SQLite database (contains feedback/experience).
   - `checkpoints/model.pt` — the current trained model checkpoint.
3. Optionally: have `data/processed/unified_stress.csv` already on Google Drive
   from a previous training run (see Step 1).

---

## Step 1 — Mount Google Drive

Your Drive is used to:
- Read `unified_stress.csv` (if you have it there from a previous run).
- Save the retrained `model.pt` so it survives after the Colab session ends.

A folder `MyDrive/StressDetection/` will be used as the root.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/StressDetection'
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
print('Drive mounted. Working folder:', DRIVE_ROOT)

## Step 2 — Clone the repository and install dependencies

This clones `anant-925/StressDetection` and installs all Python packages.
Takes about 60–90 seconds on first run.

In [ ]:
import os, subprocess

REPO_DIR = '/content/StressDetection'

if os.path.isdir(REPO_DIR):
    print('Repo already cloned — pulling latest changes.')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    print('Cloning repo…')
    subprocess.run(
        ['git', 'clone', 'https://github.com/anant-925/StressDetection.git', REPO_DIR],
        check=True
    )

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# Install Python dependencies (quiet output — takes ~60s)
!pip install -q -r requirements.txt
print('Dependencies installed.')

## Step 3 — Upload your database and checkpoint

Run the cell below. A file-picker will appear.
Upload **both** of these files from your local machine:

| File | Where to find it on your machine |
|---|---|
| `stress_detection.db` | Root of the project folder |
| `model.pt` | `checkpoints/model.pt` |

> **Tip:** You can upload them one at a time — just run the cell twice, or
> select both files at once in the picker.

In [ ]:
from google.colab import files as colab_files
import shutil, os

print('Select your stress_detection.db AND model.pt files to upload…')
uploaded = colab_files.upload()  # opens file picker

# Place each uploaded file in the correct location
for filename, content in uploaded.items():
    if filename == 'stress_detection.db':
        dest = f'{REPO_DIR}/stress_detection.db'
        with open(dest, 'wb') as f:
            f.write(content)
        print(f'  Saved database → {dest}')
    elif filename == 'model.pt':
        os.makedirs(f'{REPO_DIR}/checkpoints', exist_ok=True)
        dest = f'{REPO_DIR}/checkpoints/model.pt'
        with open(dest, 'wb') as f:
            f.write(content)
        print(f'  Saved checkpoint → {dest}')
    else:
        print(f'  Unexpected file: {filename} — ignored')

# Verify
db_ok  = os.path.isfile(f'{REPO_DIR}/stress_detection.db')
ckpt_ok = os.path.isfile(f'{REPO_DIR}/checkpoints/model.pt')
print()
print(f'stress_detection.db present: {db_ok}')
print(f'checkpoints/model.pt present: {ckpt_ok}')
if not db_ok or not ckpt_ok:
    raise RuntimeError('One or both required files are missing — please re-run this cell and upload both.')

## Step 4 — (Optional) Link the original training dataset

If `unified_stress.csv` is on your Google Drive from a previous training run,
this cell creates a symlink so `retrain.py` can use it for regularisation
(prevents catastrophic forgetting of the original training data).

**Skip this cell if you don't have the CSV** — retraining on feedback only
still works, but the model may drift slightly from its original behaviour.

In [ ]:
import os

DRIVE_CSV = f'{DRIVE_ROOT}/processed/unified_stress.csv'
LOCAL_CSV = f'{REPO_DIR}/data/processed/unified_stress.csv'

if os.path.isfile(DRIVE_CSV):
    os.makedirs(os.path.dirname(LOCAL_CSV), exist_ok=True)
    if not os.path.exists(LOCAL_CSV):
        os.symlink(DRIVE_CSV, LOCAL_CSV)
    print(f'CSV linked: {DRIVE_CSV} → {LOCAL_CSV}')
    USE_CSV = LOCAL_CSV
else:
    print('unified_stress.csv not found on Drive — will retrain on feedback samples only.')
    USE_CSV = None

print('USE_CSV =', USE_CSV)

## Step 5 — Configure retrain parameters

Edit the variables below to customise the run.
The defaults are sensible for most use-cases.

In [ ]:
# ── Retrain configuration ────────────────────────────────────────────────────

CHECKPOINT_PATH = f'{REPO_DIR}/checkpoints/model.pt'   # input checkpoint
OUTPUT_PATH     = f'{REPO_DIR}/checkpoints/model.pt'   # overwrite in-place
DB_PATH         = f'{REPO_DIR}/stress_detection.db'    # SQLite feedback DB
DATA_PATH       = USE_CSV                              # None = feedback only

EPOCHS          = 3      # fine-tuning epochs (3 is usually enough)
BATCH_SIZE      = 32     # reduce to 16 if you get CUDA out-of-memory errors
LEARNING_RATE   = 5e-4   # keep lower than initial training LR
MIN_FEEDBACK    = 10     # skip retraining if fewer than this many feedback rows
DEVICE          = 'cuda' # 'cuda' uses the free T4 GPU; change to 'cpu' if needed

# ─────────────────────────────────────────────────────────────────────────────
print('Configuration:')
print(f'  checkpoint : {CHECKPOINT_PATH}')
print(f'  output     : {OUTPUT_PATH}')
print(f'  database   : {DB_PATH}')
print(f'  csv data   : {DATA_PATH}')
print(f'  epochs     : {EPOCHS}')
print(f'  batch size : {BATCH_SIZE}')
print(f'  lr         : {LEARNING_RATE}')
print(f'  min_feedback: {MIN_FEEDBACK}')
print(f'  device     : {DEVICE}')

## Step 6 — Inspect feedback data

This cell shows how many feedback rows are in your database before retraining.
You need at least `MIN_FEEDBACK` rows (default 10) to proceed.

In [ ]:
import sys, os
sys.path.insert(0, REPO_DIR)

from database.feedback import FeedbackStore

store = FeedbackStore(DB_PATH)
total_feedback = store.get_feedback_count()
experience_rows = store.get_experience_for_training(min_samples=1)
store.close()

print(f'Total feedback events : {total_feedback}')
print(f'Experience replay rows: {len(experience_rows)}')

if total_feedback == 0:
    print()
    print('⚠️  No feedback found in the database.')
    print('   Run the app for a while, submit some feedback via the UI,')
    print('   then come back and rerun from Step 3.')
elif total_feedback < MIN_FEEDBACK:
    print()
    print(f'⚠️  Only {total_feedback} feedback rows — need at least {MIN_FEEDBACK}.')
    print(f'   Either lower MIN_FEEDBACK in Step 5, or collect more feedback first.')
else:
    print()
    print(f'✅  Enough feedback to retrain. Proceed to Step 7.')

## Step 7 — Run retraining

This is the main training step. It fine-tunes the model using reward-weighted
loss over your collected feedback, optionally mixed with the original data.

Expected runtime on a **T4 GPU**: ~2–5 minutes for 3 epochs.

In [ ]:
import sys, os
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

from training.retrain import retrain

retrain(
    checkpoint_path = CHECKPOINT_PATH,
    output_path     = OUTPUT_PATH,
    data_path       = DATA_PATH,
    db_path         = DB_PATH,
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    lr              = LEARNING_RATE,
    min_feedback    = MIN_FEEDBACK,
    device_str      = DEVICE,
)

print('\n✅  Retraining complete!')

## Step 8 — Save updated checkpoint to Google Drive

The retrained `model.pt` is copied to `MyDrive/StressDetection/checkpoints/`
so it persists after the session ends.

In [ ]:
import shutil, os

DRIVE_CKPT = f'{DRIVE_ROOT}/checkpoints/model.pt'

shutil.copy2(OUTPUT_PATH, DRIVE_CKPT)
size_mb = os.path.getsize(DRIVE_CKPT) / 1e6

print(f'Checkpoint saved to Google Drive:')
print(f'  {DRIVE_CKPT}  ({size_mb:.1f} MB)')
print()
print('The file is safe — your Colab session can now be closed.')

## Step 9 — Download the retrained model to your local machine

This triggers a browser download of the updated `model.pt`.
Place it in `checkpoints/model.pt` on your local machine, then restart
the API server:

```bash
uvicorn api.main:app --host 0.0.0.0 --port 8000
```

The server automatically loads the new weights and decision threshold on startup.

In [ ]:
from google.colab import files as colab_files
import os

if os.path.isfile(OUTPUT_PATH):
    print(f'Downloading {OUTPUT_PATH} …')
    colab_files.download(OUTPUT_PATH)
    print('Download started — check your browser Downloads folder.')
else:
    print('model.pt not found — did Step 7 complete successfully?')

---
## Tips & troubleshooting

| Symptom | Fix |
|---|---|
| `CUDA out of memory` | Set `BATCH_SIZE = 16` in Step 5 and re-run Step 7. |
| `Not enough feedback` | Lower `MIN_FEEDBACK` in Step 5, or collect more feedback via the app. |
| Session timed out before finish | The checkpoint on Drive is only written in Step 8. Re-run from Step 3. |
| Model feels worse after retrain | The original CSV wasn't used. Re-run Step 4 with the CSV on Drive, then redo Steps 5–9. |
| Want to retrain again later | Re-run from Step 3 (upload fresh DB + current model.pt). Drive backup lets you rollback. |

### How to roll back to the previous model
Before retraining, Drive already holds the previous `model.pt`.
Open [drive.google.com](https://drive.google.com), right-click the file, and choose
**Manage versions** to restore an older version.

### Running on CPU instead of GPU
If the T4 quota is exhausted, change `DEVICE = 'cpu'` in Step 5.
Training will take 5–15 minutes instead of 2–5 minutes, but works fine.